# Zadanie 5: programowanie genetyczne i regresja symboliczna

Termin realizacji: 18 maja 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zmodyfikuj przykład `pysr_demo.ipynb` tak, aby uczył się funkcji $f(x) = 2.2\sin(x_0 + 2 x_1) - x_5^2 - 3$ której dziedziną jest $\mathbb{R}^6$. Uczenie ma się odbywać w oparciu o 200 wylosowanych z dziedziny próbek (między -5 a 5).
2. Zanotuj wzory trzech rozwiązań o najwyższej wartości `score` oraz rozwiązanie `best` dla następujących zestawów ustawień:
   1. `binary_operators=["+", "*"], unary_operators=["cos", "exp", "sin"], maxsize=20`,
   2. `binary_operators=["+", "*", "-", "^"], unary_operators=["cos", "exp", "sin", "log"], maxsize=30`, (dodaj ograniczenie dla argumentów operatora "^": [https://astroautomata.com/PySR/v1.5.9/operators#custom](https://astroautomata.com/PySR/v1.5.9/operators#custom) ).
   3. `binary_operators=["+", "*", "-", "^"], unary_operators=["exp", "sin"], maxsize=15`.
3. Powtórz eksperymenty z zadania na 3.0 po dodaniu szumu do próbek z funkcji $f$ (rozkład normalny o średniej 0 i odchyleniu standardowym 0.5)

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Dodaj do porównania dopasowanie oparte o próbki losowane w szerszym zakresie (między -15 a 15) oraz wyższy poziom szumu (odchylenie standardowe równe 2 oraz 5).

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zamień funkcję $f$ na $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$ gdzie $p(i)$ oznacza $i$-tą liczbę pierwszą. Uwzględnij `p` jako dodatkowy operator unarny analogicznie do przykładu "Julia packages and types" z notatnika `pysr_demo.ipynb`. Powtórz eksperymenty opisane w zadaniach na 3.0 i 4.0.


## Importy

In [1]:
import numpy as np
import sympy
from pysr import PySRRegressor


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
default_pysr_params = dict(
    populations=50,
    model_selection="best",
    niterations=100,
)


In [3]:
SAFE_POW = "safe_pow(x, y) = (x >= 0 || isinteger(y)) ? x ^ y : convert(typeof(x), NaN)"
safe_pow_sympy = {"safe_pow": lambda x, y: x**y}


## Funkcja pomocnicza do wyświetlania wyników

In [4]:
def show_results(model):
    display(model.equations_.nlargest(3, 'score')[['complexity', 'loss', 'score', 'equation']])
    print('best:', model.sympy())


## 3.0 — Dane bez szumu, zakres [-5, 5]

Funkcja: $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - 3$, dziedzina $\mathbb{R}^6$, 200 próbek.

In [5]:
np.random.seed(0)
X = np.random.uniform(-5, 5, (200, 6))
y = 2.2 * np.sin(X[:, 0] + 2 * X[:, 1]) - X[:, 5]**2 - 3


### Konfiguracja 1 — `binary=["+","*"]`, `unary=["cos","exp","sin"]`, `maxsize=20`

In [6]:
model1 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1.fit(X, y)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
[ Info: Started!



Expressions evaluated per second: 6.390e+05
Progress: 3515 / 5000 total iterations (70.300%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  0.000e+00  y = -11.158
3           5.546e+01  1.239e-03  y = x₂ + -11.311
4           5.152e+01  7.370e-02  y = cos(x₅) + -10.987
5           6.759e+00  2.031e+00  y = (x₅ * -1.1932) * x₅
7           2.205e+00  5.600e-01  y = (x₅ * (x₅ * -0.98155)) + -3.163
9           2.197e+00  1.884e-03  y = ((x₅ + 0.033224) * (x₅ * -0.9792)) + -3.1777
11          2.173e+00  5.516e-03  y = (x₀ * 0.059189) + (((x₅ * x₅) * -0.98035) + -3.1557)
12          2.106e+00  3.128e-02  y = ((sin(x₀) * -0.43337) + ((x₅ * x₅) * -0.97735)) + -3.1...
                                      968
13          2.050e+00  2.710e-02  y = ((x₅ * (x₅ * -0.97888)) + -3.08

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_142956_w6Jztd/hall_of_fame.csv


In [7]:
show_results(model1)


,complexity,loss,score,equation
3,5,6.758767,2.031154,(x5 * -1.1931673) * x5
4,7,2.205166,0.560019,(x5 * (x5 * -0.9815521)) + -3.1630116
14,19,0.955366,0.266378,(cos(cos((exp(sin(x0)) * -0.61584556) + x1)) *...


best: x5*(-0.99212056)*x5 + cos(cos(x1 + exp(sin(x0))*(-0.61584556)))*6.8843594 - 8.279847


### Konfiguracja 2 — `binary=["+","*","-","^"]`, `unary=["cos","exp","sin","log"]`, `maxsize=30`

Ograniczenie dla `^`: prawy argument ma maksymalną złożoność 1.

In [8]:
model2 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model2.fit(X, y)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 4.150e+05
Progress: 2299 / 5000 total iterations (45.980%)
════════════════════════════════════════════════════════════════════════════════════════════════════


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  0.000e+00  y = -11.158
3           5.537e+01  2.049e-03  y = -11.021 - x₅
4           5.152e+01  7.208e-02  y = cos(x₅) + -10.987
5           2.224e+00  3.143e+00  y = -3.0128 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = ((x₅ * x₅) - -3.2223) * -0.98156
9           2.153e+00  1.189e-02  y = (x₅ * ((x₄ * 0.03329) - x₅)) - 3.0191
10          2.134e+00  9.020e-03  y = (-3.0123 - (x₅ * x₅)) - (sin(x₀) * 0.41059)
11          2.074e+00  2.840e-02  y = (-2.912 - (x₅ * x₅)) + cos(cos(x₁) + x₁)
12          6.617e-01  1.143e+00  y = (sin((x₁ + x₁) + x₀) - 3.007) - (x₅ * x₅)
14          6.561e-01  4.262e-03  y = (sin(x₁ + (x₁ + x₀)) + -3.089) - ((x₅ * 0.98993) * x₅)
16          8.505e-13  1.369e+01  y = ((cos((x₁ + x₁) + (x₀ + -1.5708)) * 2.2) + -3) - (x₅ *...
                                       x₅)
20          7.041e-13  4.7

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143009_YLJBt8/hall_of_fame.csv


In [9]:
show_results(model2)


,complexity,loss,score,equation
10,14,8.463985e-13,27.239924,((sin((x1 + x1) + x0) * 2.2) - (x5 * x5)) + -3...
3,5,2.224043e+00,3.142667,-3.0129461 - (x5 * x5)
8,12,6.616987e-01,1.142550,(sin((x0 + x1) + x1) - 3.0070133) - (x5 * x5)


best: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002


### Konfiguracja 3 — `binary=["+","*","-","^"]`, `unary=["exp","sin"]`, `maxsize=15`

In [10]:
model3 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model3.fit(X, y)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.560e+01  0.000e+00  y = -11.158
3           5.537e+01  2.049e-03  y = -11.021 - x₅
4           5.462e+01  1.364e-02  y = -11.157 - sin(x₀)
5           2.224e+00  3.201e+00  y = -3.0129 - (x₅ * x₅)
7           2.205e+00  4.262e-03  y = ((x₅ * -0.98154) * x₅) - 3.1631
9           2.153e+00  1.189e-02  y = -3.0191 - (x₅ * (x₅ + (x₄ * -0.03329)))
10          2.106e+00  2.231e-02  y = (sin(x₁ * -2.0613) + -3.0706) - (x₅ * x₅)
11          2.067e+00  1.866e-02  y = -3.0645 - (sin(sin(x₁ * 2.0553)) + (x₅ * x₅))
12          6.617e-01  1.139e+00  y = (sin(x₁ + (x₀ + x₁)) + -3.007) - (x₅ * x₅)
13          5.722e-01  1.452e-01  y = exp(sin(x₀ + (x₁ * 1.9959))) + (-4.2504 - (x₅ * x₅))
14          8.464e-13  2.724e+01  y = ((sin((x₁ + x₁) + x₀) * 2.2) - (x₅ * x₅)) + -3
─────────────────────────────────────────────────────────────────────────────

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin']"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143019_O2Ef4k/hall_of_fame.csv


In [11]:
show_results(model3)


,complexity,loss,score,equation
10,14,8.463985e-13,27.239601,((sin((x1 + x1) + x0) * 2.2) - (x5 * x5)) + -3...
3,5,2.224043e+00,3.201109,-3.0128849 - (x5 * x5)
8,12,6.616987e-01,1.138992,(sin(x1 + (x0 + x1)) + -3.0070133) - (x5 * x5)


best: -x5*x5 + sin(x0 + x1 + x1)*2.2 - 3.0000002


## 3.0 — Dane z szumem $\mathcal{N}(0,\,0.5^2)$, zakres [-5, 5]

In [12]:
np.random.seed(0)
X_n05 = np.random.uniform(-5, 5, (200, 6))
y_n05 = 2.2 * np.sin(X_n05[:, 0] + 2 * X_n05[:, 1]) - X_n05[:, 5]**2 - 3 + np.random.normal(0, 0.5, 200)


### Konfiguracja 1 (szum 0.5)

In [13]:
model1_n05 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_n05.fit(X_n05, y_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 7.910e+05
Progress: 4329 / 5000 total iterations (86.580%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  0.000e+00  y = -11.092
3           5.560e+01  5.008e-04  y = x₂ + -11.246
4           5.159e+01  7.493e-02  y = cos(x₅) + -10.922
5           6.855e+00  2.018e+00  y = (x₅ * -1.188) * x₅
7           2.449e+00  5.148e-01  y = (x₅ * (x₅ * -0.97982)) + -3.1117
9           2.434e+00  2.937e-03  y = (x₅ * ((x₅ * -0.97674) + -0.042771)) + -3.1309
11          2.363e+00  1.485e-02  y = (x₅ * ((x₅ * -0.97839) + (x₄ * 0.036652))) + -3.1303
12          2.332e+00  1.324e-02  y = ((sin(x₁ * 2.0193) + (x₅ * x₅)) * -0.97759) + -3.1949
13          2.292e+00  1.707e-02  y = (((x₅ * x₅) + sin(sin(x₁ * 2.0148))) * -0.97868) + -3....
                 

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143025_4pHJR9/hall_of_fame.csv


In [14]:
show_results(model1_n05)


,complexity,loss,score,equation
3,5,6.855120,2.018258,(x5 * -1.1879798) * x5
11,16,0.203464,1.359571,((x5 * (x5 * -0.9997855)) + (sin((x1 * 1.98074...
9,14,0.882724,0.954339,sin(x0 + (x1 * 1.9837806)) + ((x5 * (x5 * -0.9...


best: x5*x5*(-0.9997855) + sin(x0 + x1*1.980745)*2.2228205 - 2.9419146


### Konfiguracja 2 (szum 0.5)

In [15]:
model2_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model2_n05.fit(X_n05, y_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 5.260e+05
Progress: 2868 / 5000 total iterations (57.360%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  0.000e+00  y = -11.093
3           5.528e+01  3.373e-03  y = -10.955 - x₅
4           5.159e+01  6.919e-02  y = cos(x₅) + -10.922
5           2.471e+00  3.039e+00  y = -2.9475 - (x₅ * x₅)
7           2.449e+00  4.585e-03  y = (x₅ * (x₅ * -0.97982)) + -3.1117
9           2.389e+00  1.235e-02  y = (((x₄ * 0.035916) - x₅) * x₅) + -2.9541
10          2.360e+00  1.222e-02  y = -3.0128 - (sin(x₁ * 2.025) + (x₅ * x₅))
11          2.292e+00  2.917e-02  y = (-2.9825 - (x₅ * x₅)) - (sin(x₁) * cos(x₁))
12          8.897e-01  9.462e-01  y = (-2.9439 - (x₅ * x₅)) - sin((x₁ * -1.9819) - x₀)
14          2.035e-01  7.377e-01  y = (-2.9403 - (x₅ * 

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143031_eOBvha/hall_of_fame.csv


In [16]:
show_results(model2_n05)


,complexity,loss,score,equation
3,5,2.471086,3.038596,-2.9474647 - (x5 * x5)
8,12,0.889746,0.946208,(-2.943917 - (x5 * x5)) - sin((x1 * -1.9819144...
9,14,0.203466,0.737718,(-2.9401376 - (x5 * x5)) - (sin((x1 * -1.98069...


best: -x5*x5 - 2.222888*sin(-x0 + x1*(-1.9806954)) - 2.9401376


### Konfiguracja 3 (szum 0.5)

In [17]:
model3_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model3_n05.fit(X_n05, y_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 7.850e+05
Progress: 4602 / 5000 total iterations (92.040%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.566e+01  0.000e+00  y = -11.092
3           5.528e+01  3.373e-03  y = -10.956 - x₅
4           5.467e+01  1.109e-02  y = -11.093 - sin(x₀)
5           2.471e+00  3.097e+00  y = -2.9476 - (x₅ * x₅)
7           2.449e+00  4.585e-03  y = ((x₅ * x₅) * -0.97982) - 3.1117
9           2.389e+00  1.235e-02  y = -2.9541 - (x₅ * (x₅ - (x₄ * 0.035946)))
10          2.360e+00  1.222e-02  y = -3.0128 - ((x₅ * x₅) - sin(x₁ * -2.025))
11          2.318e+00  1.800e-02  y = (-3.006 - (x₅ * x₅)) - sin(sin(x₁ * 2.0184))
12          8.930e-01  9.537e-01  y = (sin(x₁ + (x₁ + x₀)) + -2.9415) - (x₅ * x₅)
13          8.046e-01  1.043e-01  y = exp(sin((x₁ + x₁) + x

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin']"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143040_PAQruO/hall_of_fame.csv


In [18]:
show_results(model3_n05)


,complexity,loss,score,equation
3,5,2.471086,3.096697,-2.9475615 - (x5 * x5)
10,14,0.212364,1.332022,((sin((x1 + x0) + x1) * 2.2170868) - (x5 * x5)...
8,12,0.889746,0.957383,(sin(x0 + (x1 * 1.9819064)) - (x5 * x5)) + -2....


best: -x5*x5 + sin(x0 + x1 + x1)*2.2170868 - 2.934413


## 4.0 — Zakres [-15, 15], szum $\sigma=2$

In [19]:
np.random.seed(0)
X_w2 = np.random.uniform(-15, 15, (200, 6))
y_w2 = 2.2 * np.sin(X_w2[:, 0] + 2 * X_w2[:, 1]) - X_w2[:, 5]**2 - 3 + np.random.normal(0, 2, 200)


### Konfiguracja 1 (zakres [-15,15], σ=2)

In [20]:
model1_w2 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_w2.fit(X_w2, y_w2)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 7.490e+05
Progress: 3961 / 5000 total iterations (79.220%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  0.000e+00  y = -76.023
3           4.336e+03  1.630e-02  y = x₂ + -76.478
5           9.261e+00  3.074e+00  y = (x₅ * -1.0195) * x₅
7           5.635e+00  2.484e-01  y = ((x₅ * -0.99853) * x₅) + -2.8222
9           5.631e+00  3.749e-04  y = (((x₅ * -0.99872) + 0.0077422) * x₅) + -2.812
11          5.553e+00  6.948e-03  y = (((x₅ * x₅) + sin(exp(x₄))) * -0.9984) + -2.8637
12          5.147e+00  7.585e-02  y = ((sin(x₀ * -1.4498) + (x₅ * x₅)) * -0.9984) + -2.8637
14          5.132e+00  1.488e-03  y = (sin((x₀ * 1.455) + 0.18589) + (x₅ * (x₅ * -0.99853)))...
                                       + -2.8461
15          4.869e+00  5.268

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143046_ppUdL6/hall_of_fame.csv


In [21]:
show_results(model1_w2)


,complexity,loss,score,equation
2,5,9.260938,3.074411,(x5 * -1.0195066) * x5
3,7,5.635046,0.248400,((x5 * -0.99853265) * x5) + -2.822222
6,12,5.147461,0.075854,((sin(x0 * -1.4498069) + (x5 * x5)) * -0.99839...


best: x5*(-0.99853265)*x5 - 2.822222


### Konfiguracja 2 (zakres [-15,15], σ=2)

In [22]:
model2_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model2_w2.fit(X_w2, y_w2)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 4.560e+05
Progress: 2463 / 5000 total iterations (49.260%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  0.000e+00  y = -76.023
3           4.336e+03  1.630e-02  y = x₂ + -76.481
5           5.645e+00  3.322e+00  y = -2.7146 - (x₅ * x₅)
7           5.635e+00  8.621e-04  y = (-2.8265 - (x₅ * x₅)) * 0.99853
9           5.554e+00  7.223e-03  y = -2.7191 - ((x₅ * x₅) + sin(exp(x₄)))
10          5.159e+00  7.382e-02  y = (sin(x₀ * 1.4504) - (x₅ * x₅)) + -2.7464
12          4.115e+00  1.131e-01  y = (-2.7253 - (x₅ * x₅)) + sin((x₀ + x₁) + x₁)
14          3.396e+00  9.602e-02  y = ((sin((x₁ + x₀) + x₁) * 2.3008) + -2.7391) - (x₅ * x₅)
16          3.380e+00  2.304e-03  y = ((sin(((x₁ + -0.07202) + x₁) + x₀) * 2.2989) - (x₅ * x...
            

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143052_Kc7Myg/hall_of_fame.csv


In [23]:
show_results(model2_w2)


,complexity,loss,score,equation
2,5,5.644776,3.321948,-2.7146325 - (x5 * x5)
8,14,3.395651,0.156221,(sin(x1 + (x1 + x0)) * 2.3006766) + (-2.739261...
6,12,4.114574,0.113101,(-2.725299 - (x5 * x5)) + sin((x0 + x1) + x1)


best: -x5*x5 + sin(x0 + x1 + x1)*2.3006766 - 2.7392614


### Konfiguracja 3 (zakres [-15,15], σ=2)

In [24]:
model3_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model3_w2.fit(X_w2, y_w2)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.482e+03  0.000e+00  y = -76.023
3           4.336e+03  1.630e-02  y = x₂ + -76.478
5           5.645e+00  3.322e+00  y = -2.7146 - (x₅ * x₅)
7           5.635e+00  8.623e-04  y = (-2.8262 - (x₅ * x₅)) * 0.99853
9           5.548e+00  7.826e-03  y = (-2.6458 - (x₅ * x₅)) - sin(exp(x₄))
10          5.159e+00  7.261e-02  y = (-2.7463 - (x₅ * x₅)) - sin(x₀ * -1.4504)
12          4.108e+00  1.139e-01  y = (-2.7218 - (x₅ * x₅)) - sin((x₁ * -2.0095) - x₀)
14          4.094e+00  1.706e-03  y = (-2.7218 - (x₅ * x₅)) - sin(((x₁ * -2.0095) - x₀) - -0...
                                      .11062)
───────────────────────────────────────────────────────────────────────────────────────────────────


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin']"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143102_PAQruO/hall_of_fame.csv


In [25]:
show_results(model3_w2)


,complexity,loss,score,equation
2,5,5.644776,3.321948,-2.7146323 - (x5 * x5)
6,12,4.107902,0.113913,(-2.7218335 - (x5 * x5)) - sin((x1 * -2.009524...
5,10,5.158975,0.072615,(-2.746339 - (x5 * x5)) - sin(x0 * -1.4504223)


best: -x5*x5 - 2.7146323


## 4.0 — Zakres [-15, 15], szum $\sigma=5$

In [26]:
np.random.seed(0)
X_w5 = np.random.uniform(-15, 15, (200, 6))
y_w5 = 2.2 * np.sin(X_w5[:, 0] + 2 * X_w5[:, 1]) - X_w5[:, 5]**2 - 3 + np.random.normal(0, 5, 200)


### Konfiguracja 1 (zakres [-15,15], σ=5)

In [27]:
model1_w5 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin"],
    maxsize=20,
    **default_pysr_params,
)
model1_w5.fit(X_w5, y_w5)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 8.100e+05
Progress: 4388 / 5000 total iterations (87.760%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.489e+03  0.000e+00  y = -75.633
3           4.345e+03  1.610e-02  y = x₂ + -76.091
5           2.662e+01  2.547e+00  y = (x₅ * -1.0161) * x₅
7           2.375e+01  5.716e-02  y = (x₅ * (x₅ * -0.99738)) + -2.5136
9           2.374e+01  2.401e-04  y = ((x₅ + 0.012771) * (x₅ * -0.99708)) + -2.5309
10          2.335e+01  1.650e-02  y = ((x₅ * x₅) + (sin(x₁) + 2.4112)) * -0.99884
11          2.326e+01  3.815e-03  y = ((exp(sin(x₅)) + (x₅ * x₅)) * -0.99596) + -1.4113
12          2.252e+01  3.207e-02  y = (sin(x₀ * 1.4636) + (x₅ * (x₅ * -0.9973))) + -2.5551
14          2.204e+01  1.088e-02  y = ((x₅ * x₅) * -0.99924) + (-2.4296 + (sin(x₀ * 13.369)

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143107_6I5wdN/hall_of_fame.csv


In [28]:
show_results(model1_w5)


,complexity,loss,score,equation
2,5,26.622316,2.547511,(x5 * -1.01606) * x5
3,7,23.746544,0.057156,(x5 * (x5 * -0.9973819)) + -2.513649
7,12,22.523647,0.032067,(sin(x0 * 1.4636205) + (x5 * (x5 * -0.99730337...


best: x5*(-1.01606)*x5


### Konfiguracja 2 (zakres [-15,15], σ=5)

In [29]:
model2_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model2_w5.fit(X_w5, y_w5)



Expressions evaluated per second: 4.120e+05
Progress: 2161 / 5000 total iterations (43.220%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.489e+03  0.000e+00  y = -75.634
3           4.345e+03  1.610e-02  y = x₂ + -76.085
5           2.378e+01  2.604e+00  y = -2.322 - (x₅ * x₅)
7           2.375e+01  6.469e-04  y = (x₅ * (x₅ * -0.99738)) + -2.5143
8           2.335e+01  1.673e-02  y = (-2.3232 - sin(x₁)) - (x₅ * x₅)
9           2.333e+01  9.209e-04  y = -1.1105 - (exp(sin(x₅)) + (x₅ * x₅))
10          2.285e+01  2.076e-02  y = (sin(x₅ * -6.6832) - (x₅ * x₅)) - 2.3896
12          2.265e+01  4.425e-03  y = (-2.2846 - (x₅ * x₅)) + cos((x₁ - x₅) - -0.80223)
13          2.243e+01  9.667e-03  y = (sin(-0.62138 - (x₁ - cos(x₅))) + -2.3172) - (x₅ * x₅)
15          2.237e+01  1.303e-03

/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143113_SBn4Bs/hall_of_fame.csv


In [30]:
show_results(model2_w5)


,complexity,loss,score,equation
2,5,23.777332,2.604020,-2.3219779 - (x5 * x5)
6,10,22.556126,0.033780,(sin(x0 * 1.4643761) - (x5 * x5)) + -2.3566928
13,19,20.742260,0.017744,sin(x1 + (x0 + x1)) + ((-2.3522458 - (x5 * x5)...


best: -x5*x5 - 2.3219779


### Konfiguracja 3 (zakres [-15,15], σ=5)

In [31]:
model3_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings=safe_pow_sympy,
    **default_pysr_params,
)
model3_w5.fit(X_w5, y_w5)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 7.660e+05
Progress: 4553 / 5000 total iterations (91.060%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.489e+03  0.000e+00  y = -75.634
3           4.345e+03  1.610e-02  y = x₂ + -76.091
5           2.378e+01  2.604e+00  y = -2.3215 - (x₅ * x₅)
7           2.375e+01  6.469e-04  y = (x₅ * (x₅ * -0.99738)) - 2.5135
8           2.335e+01  1.673e-02  y = (-2.3233 - (x₅ * x₅)) - sin(x₁)
9           2.333e+01  9.209e-04  y = -1.1102 - (exp(sin(x₅)) + (x₅ * x₅))
10          2.256e+01  3.378e-02  y = -2.3568 - (sin(x₀ * -1.4643) + (x₅ * x₅))
12          2.204e+01  1.157e-02  y = -2.3429 - ((x₅ * x₅) - sin(x₀ - (x₁ * -2.0302)))
14          2.194e+01  2.204e-03  y = -2.3287 - ((x₅ * x₅) - sin((x₀ - (x₁ * -2.0391)) + -0....
                             

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.489e+03  0.000e+00  y = -75.634
3           4.345e+03  1.610e-02  y = x₂ + -76.091
5           2.378e+01  2.604e+00  y = -2.3215 - (x₅ * x₅)
7           2.375e+01  6.469e-04  y = (x₅ * (x₅ * -0.99738)) - 2.5135
8           2.335e+01  1.673e-02  y = (-2.3233 - (x₅ * x₅)) - sin(x₁)
9           2.333e+01  9.209e-04  y = -1.1102 - (exp(sin(x₅)) + (x₅ * x₅))
10          2.256e+01  3.378e-02  y = -2.3568 - (sin(x₀ * -1.4643) + (x₅ * x₅))
12          2.204e+01  1.157e-02  y = (-2.3424 - (x₅ * x₅)) + sin(x₀ - (x₁ * -2.0302))
14          2.194e+01  2.204e-03  y = -2.3287 - ((x₅ * x₅) - sin((x₀ - (x₁ * -2.0391)) + -0....
                                      28811))
15          2.166e+01  1.279e-02  y = -2.3437 - ((x₅ * x₅) + (sin(x₁) - sin(x₀ - (x₁ * -2.02...
                                      86))))
─────────────────────────────────────

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin']"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143124_PAQruO/hall_of_fame.csv


In [32]:
show_results(model3_w5)


,complexity,loss,score,equation
2,5,23.777332,2.604020,-2.321463 - (x5 * x5)
6,10,22.556124,0.033780,-2.3567507 - (sin(x0 * -1.4643459) + (x5 * x5))
4,8,23.352629,0.016728,(-2.3232915 - (x5 * x5)) - sin(x1)


best: -x5*x5 - 2.321463


## 5.0 — Operator liczb pierwszych $p(i)$

Funkcja: $f(x) = 2.2\sin(x_0 + 2x_1) - x_5^2 - p(\lfloor x_0 \rfloor)$

### Instalacja Primes.jl i definicja operatora `p`

In [33]:
from pysr import jl

jl.seval("""
import Pkg
Pkg.add(\"Primes\")
""")


   Resolving package versions...
     Project No packages added to or removed from `/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/julia_env/Project.toml`
    Manifest No packages added to or removed from `/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/julia_env/Manifest.toml`


In [34]:
jl.seval("using Primes: prime")


In [35]:
jl.seval("""
function p(i::T) where T
    if 0.5 < i < 1000
        return T(prime(round(Int, i)))
    else
        return T(NaN)
    end
end
""")


p (generic function with 1 method)

Definicja `sympy_p` oraz pomocnicza funkcja Pythonowa do generowania próbek.

In [36]:
class sympy_p(sympy.Function):
    pass

def p_python(x0_val):
    i = int(np.floor(x0_val))
    if 1 <= i <= 999:
        return float(sympy.prime(i))
    return np.nan


In [37]:
def make_data_prime(low, high, n=200, noise_std=0.0, seed=0):
    rng = np.random.default_rng(seed)
    # generujemy więcej próbek, bo część x0 daje NaN
    X_raw = rng.uniform(low, high, (n * 6, 6))
    p_vals = np.array([p_python(x) for x in X_raw[:, 0]])
    y_raw = 2.2 * np.sin(X_raw[:, 0] + 2 * X_raw[:, 1]) - X_raw[:, 5]**2 - p_vals
    if noise_std > 0:
        y_raw += rng.normal(0, noise_std, len(y_raw))
    mask = ~np.isnan(y_raw)
    return X_raw[mask][:n], y_raw[mask][:n]


### 5.0 — Bez szumu, zakres [-5, 5]

In [38]:
X_p, y_p = make_data_prime(-5, 5)
print(f'Próbek: {len(y_p)}')


Próbek: 200


### Konfiguracja 1 z operatorem `p` (bez szumu)

In [39]:
model1_p = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p.fit(X_p, y_p)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 5.890e+05
Progress: 3458 / 5000 total iterations (69.160%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.486e+01  0.000e+00  y = -11.861
4           5.058e+01  2.706e-02  y = cos(x₅) + -11.678
5           1.433e+01  1.261e+00  y = x₅ * (x₅ * -1.2969)
7           3.456e+00  7.112e-01  y = (x₀ + (x₅ * x₅)) * -1.0779
9           2.617e+00  1.391e-01  y = (x₀ * -1.4628) + (x₅ * (x₅ * -0.9778))
10          2.531e+00  3.363e-02  y = ((x₅ * x₅) + p(x₀ * 0.8964)) * -0.97731
12          2.380e+00  3.062e-02  y = (p((x₀ * 0.98267) + -0.4172) + (x₅ * x₅)) * -0.98774
15          2.189e+00  2.787e-02  y = (cos(cos(x₁ + 1.041)) * (x₀ * -1.8986)) + (x₅ * (x₅ * ...
                                      -0.98279))
16          9.839e-01  7.998e-01  y = sin(x₁ +

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143140_2GQyRW/hall_of_fame.csv


In [40]:
show_results(model1_p)


,complexity,loss,score,equation
2,5,14.333841,1.260923,x5 * (x5 * -1.296927)
9,16,0.983938,0.799834,sin(x1 + (x0 + x1)) + ((x0 * -1.456332) + (x5 ...
3,7,3.456479,0.711186,(x0 + (x5 * x5)) * -1.0778502


best: x0*(-1.4489) + x5*x5*(-0.9958418) + 2.1464548*sin(x0 + x1 + x1)


### Konfiguracja 2 z operatorem `p` (bez szumu)

In [41]:
model2_p = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model2_p.fit(X_p, y_p)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 4.080e+05
Progress: 2302 / 5000 total iterations (46.040%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.486e+01  0.000e+00  y = -11.86
3           5.151e+01  3.144e-02  y = -8.8115 - x₀
4           5.058e+01  1.828e-02  y = cos(x₅) + -11.678
5           5.660e+00  2.190e+00  y = -4.263 - (x₅ * x₅)
7           2.985e+00  3.198e-01  y = (-1.2136 - (x₅ * x₅)) - x₀
9           2.581e+00  7.271e-02  y = -0.25762 - ((x₅ * x₅) + safe_pow(x₀, 1.2265))
10          2.545e+00  1.412e-02  y = ((-1.2334 - (x₅ * x₅)) + sin(x₀)) - x₀
11          2.525e+00  7.804e-03  y = x₀ + (-1.7343 - (safe_pow(x₀, 1.5038) + (x₅ * x₅)))
12          2.474e+00  2.062e-02  y = ((sin(x₀ + -0.49333) - x₀) - (x₅ * x₅)) + -1.4527
13          2.451e+00  9.229e-03  y = (-1.4078 + 

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143148_WgPEBO/hall_of_fame.csv


In [42]:
show_results(model2_p)


,complexity,loss,score,equation
3,5,5.659594,2.190194,-4.26304 - (x5 * x5)
9,14,0.950600,0.881503,"sin(x1 + (x0 + x1)) - (safe_pow(x0, 1.2676626)..."
10,16,0.286771,0.599204,(sin((x1 + x1) + x0) * 2.1523123) - (safe_pow(...


best: -(x0**1.2748233 + x5*x5) + sin(x0 + x1 + x1)*2.1523123


### Konfiguracja 3 z operatorem `p` (bez szumu)

In [43]:
model3_p = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model3_p.fit(X_p, y_p)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 6.080e+05
Progress: 3738 / 5000 total iterations (74.760%)
════════════════════════════════════════════════════════════════════════════════════════════════════


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.486e+01  0.000e+00  y = -11.86
3           5.151e+01  3.144e-02  y = -8.8112 - x₀
5           5.660e+00  1.104e+00  y = -4.263 - (x₅ * x₅)
7           2.645e+00  3.804e-01  y = (x₀ * -1.4136) - (x₅ * x₅)
9           2.581e+00  1.211e-02  y = -0.25759 - ((x₅ * x₅) + safe_pow(x₀, 1.2265))
10          2.402e+00  7.210e-02  y = (x₅ * (x₅ * -0.98467)) - p(x₀ + -0.49424)
12          2.375e+00  5.639e-03  y = ((x₀ * -1.4262) - sin(x₁ + x₁)) - (x₅ * x₅)
13          2.364e+00  4.747e-03  y = ((x₀ * -1.4248) - sin(sin(x₁ + x₁))) - (x₅ * x₅)
14          2.336e+00  1.186e-02  y = (((x₀ * -1.5244) - sin(x₁ + x₁)) - (x₅ * x₅)) * 0.9736...
                                      7
15          1.304e+00  5.828e-01  y = ((x₀ * -1.4258) - sin((x₁ + x₁) - sin(x₀))) - (x₅ * x₅...
                                      )
──────────────────────────────────

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143200_sXUFFP/hall_of_fame.csv


In [44]:
show_results(model3_p)


,complexity,loss,score,equation
2,5,5.659594,1.104241,-4.2630405 - (x5 * x5)
8,14,0.994522,0.865675,(x0 * -1.4257586) + (sin(x1 + (x0 + x1)) - (x5...
3,7,2.644633,0.380410,(x0 * -1.4136101) - (x5 * x5)


best: x0*(-1.4257586) - x5*x5 + sin(x0 + x1 + x1)


### 5.0 — Szum $\sigma=0.5$, zakres [-5, 5]

In [45]:
X_p_n05, y_p_n05 = make_data_prime(-5, 5, noise_std=0.5)


### Konfiguracja 1 z operatorem `p` (szum 0.5)

In [46]:
model1_p_n05 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_n05.fit(X_p_n05, y_p_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 5.170e+05
Progress: 3050 / 5000 total iterations (61.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.512e+01  0.000e+00  y = -11.902
4           5.090e+01  2.654e-02  y = cos(x₅) + -11.72
5           1.463e+01  1.247e+00  y = x₅ * (x₅ * -1.3003)
7           3.753e+00  6.801e-01  y = ((x₅ * x₅) + x₀) * -1.0674
9           2.893e+00  1.302e-01  y = (x₀ * -1.4638) + (x₅ * (x₅ * -0.98098))
11          2.891e+00  3.510e-04  y = ((x₅ * (x₅ * -0.98301)) + (x₀ * -1.4983)) + 0.13535
12          2.776e+00  4.037e-02  y = (((x₅ * x₅) * -0.98087) + (sin(x₀) * 2.3133)) + -4.495...
                                      2
14          2.612e+00  3.060e-02  y = (((x₅ * x₅) + sin(x₁ + x₁)) * -0.97807) + (x₀ * -1.481...
                                      

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143207_jKrjqi/hall_of_fame.csv


In [47]:
show_results(model1_p_n05)


,complexity,loss,score,equation
2,5,14.626331,1.247088,x5 * (x5 * -1.300273)
9,16,1.237752,0.742257,(x5 * (x5 * -0.9893826)) + ((x0 * -1.4578174) ...
3,7,3.724829,0.683901,((x5 * x5) + x0) * -1.0805811


best: x0*(-1.4503202) + x5*(-0.999171)*x5 + sin(x0 + x1*1.9893309)*2.163329


### Konfiguracja 2 z operatorem `p` (szum 0.5)

In [48]:
model2_p_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model2_p_n05.fit(X_p_n05, y_p_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 3.320e+05
Progress: 1876 / 5000 total iterations (37.520%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.512e+01  0.000e+00  y = -11.901
3           5.186e+01  3.052e-02  y = -8.8527 - x₀
4           5.090e+01  1.859e-02  y = cos(x₅) + -11.72
5           5.818e+00  2.169e+00  y = -4.3043 - (x₅ * x₅)
7           2.913e+00  3.459e-01  y = (x₀ * -1.4217) - (x₅ * x₅)
9           2.873e+00  6.904e-03  y = (-0.3638 - (x₅ * x₅)) - safe_pow(x₀, 1.213)
10          2.845e+00  9.705e-03  y = ((sin(x₀) + -1.2747) - (x₅ * x₅)) - x₀
11          2.831e+00  5.017e-03  y = (sin(sin(x₀)) - (x₅ * x₅)) + (-1.269 - x₀)
12          2.751e+00  2.860e-02  y = (((sin(x₁) * 0.21441) - x₅) * x₅) - safe_pow(x₀, 1.270...
                                      1)
13        

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143215_5UmtJH/hall_of_fame.csv


In [49]:
show_results(model2_p_n05)


,complexity,loss,score,equation
3,5,5.818119,2.168934,-4.304165 - (x5 * x5)
11,18,0.566741,0.730512,((cos((x1 + (x0 + 11.008982)) + x1) * 2.168232...
8,14,1.247065,0.370354,(sin(x1 + (x0 + x1)) - (x5 * x5)) - (x0 * 1.43...


best: -1.4474493*x0 - x5*x5 + cos(x0 + x1 + x1 + 11.008982)*2.1682327


### Konfiguracja 3 z operatorem `p` (szum 0.5)

In [50]:
model3_p_n05 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model3_p_n05.fit(X_p_n05, y_p_n05)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(



Expressions evaluated per second: 7.510e+05
Progress: 4398 / 5000 total iterations (87.960%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.512e+01  0.000e+00  y = -11.903
3           5.186e+01  3.052e-02  y = -8.8524 - x₀
5           5.818e+00  1.094e+00  y = -4.3042 - (x₅ * x₅)
7           2.913e+00  3.459e-01  y = (x₀ * -1.4217) - (x₅ * x₅)
9           2.873e+00  6.938e-03  y = (safe_pow(1.4261, x₀) * -1.3501) - (x₅ * x₅)
10          2.795e+00  2.758e-02  y = ((sin(x₀) + -1.8874) * 2.3046) - (x₅ * x₅)
12          2.642e+00  2.812e-02  y = sin(x₁ * -1.9981) - ((x₅ * x₅) - (x₀ * -1.4343))
13          2.627e+00  5.676e-03  y = sin(sin(x₁ * -1.9896)) - ((x₅ * x₅) - (x₀ * -1.4334))
14          8.192e-01  1.165e+00  y = ((sin((x₁ + x₀) + x₁) - x₀) * 1.4727) - (x₅ * x₅)
───────────

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.512e+01  0.000e+00  y = -11.903
3           5.186e+01  3.052e-02  y = -8.8524 - x₀
5           5.818e+00  1.094e+00  y = -4.3042 - (x₅ * x₅)
7           2.913e+00  3.459e-01  y = (x₀ * -1.4217) - (x₅ * x₅)
9           2.873e+00  6.938e-03  y = (safe_pow(1.4261, x₀) * -1.3501) - (x₅ * x₅)
10          2.795e+00  2.758e-02  y = ((sin(x₀) + -1.8874) * 2.3046) - (x₅ * x₅)
12          2.642e+00  2.812e-02  y = sin(x₁ * -1.9981) - ((x₅ * x₅) - (x₀ * -1.4343))
13          2.627e+00  5.676e-03  y = sin(sin(x₁ * -1.9896)) - ((x₅ * x₅) - (x₀ * -1.4334))
14          8.192e-01  1.165e+00  y = ((sin(x₀ + (x₁ + x₁)) - x₀) * 1.4727) - (x₅ * x₅)
───────────────────────────────────────────────────────────────────────────────────────────────────


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143227_BxYIBa/hall_of_fame.csv


In [51]:
show_results(model3_p_n05)


,complexity,loss,score,equation
8,14,0.819225,1.165212,((sin(x0 + (x1 + x1)) - x0) * 1.4726788) - (x5...
2,5,5.818119,1.093766,-4.304165 - (x5 * x5)
3,7,2.913032,0.345891,(x0 * -1.4216827) - (x5 * x5)


best: -x5*x5 + (-x0 + sin(x0 + x1 + x1))*1.4726788


### 5.0 — Zakres [-15, 15], szum $\sigma=2$

In [52]:
X_p_w2, y_p_w2 = make_data_prime(-15, 15, noise_std=2)


### Konfiguracja 1 z operatorem `p` (zakres [-15,15], σ=2)

In [53]:
model1_p_w2 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_w2.fit(X_p_w2, y_p_w2)



Expressions evaluated per second: 3.160e+05
Progress: 1486 / 5000 total iterations (29.720%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.636e+03  0.000e+00  y = -95.21
3           4.632e+03  1.532e-04  y = x₂ + -95.436
4           4.624e+03  1.210e-03  y = sin(x₂) + -95.208
5           3.854e+02  2.485e+00  y = (x₅ * x₅) * -1.1556
7           1.546e+02  4.566e-01  y = ((x₅ * x₅) + x₀) * -1.0969
8           9.320e+00  2.809e+00  y = (p(x₀) + (x₅ * x₅)) * -0.989
10          6.523e+00  1.784e-01  y = (p(x₀ + -0.50809) + (x₅ * x₅)) * -1.0011
12          6.464e+00  4.576e-03  y = ((x₅ * (x₅ + -0.037805)) + p(-0.50751 + x₀)) * -1.001
14          6.464e+00  8.941e-08  y = ((x₅ * x₅) + ((x₅ * -0.037805) + p(x₀ + -0.50751))) * ...
                                      -1.001
15      

/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143234_PEDQBo/hall_of_fame.csv


In [54]:
show_results(model1_p_w2)


,complexity,loss,score,equation
5,8,9.319632,2.808837,(p(x0) + (x5 * x5)) * -0.98899573
3,5,385.373080,2.484887,(x5 * x5) * -1.1555504
4,7,154.618360,0.456626,((x5 * x5) + x0) * -1.0968556


best: (x5*x5 + sympy_p(x0 - 0.50922304))*(-1.0011047)


### Konfiguracja 2 z operatorem `p` (zakres [-15,15], σ=2)

In [55]:
model2_p_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model2_p_w2.fit(X_p_w2, y_p_w2)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.710e+05
Progress: 1364 / 5000 total iterations (27.280%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.636e+03  0.000e+00  y = -95.206
3           4.540e+03  1.017e-02  y = -87.118 - x₀
5           1.952e+02  1.573e+00  y = -20.77 - (x₅ * x₅)
7           1.644e+01  1.237e+00  y = (x₀ * -2.7182) - (x₅ * x₅)
8           8.644e+00  6.427e-01  y = (p(x₀) * -0.94168) - (x₅ * x₅)
10          6.481e+00  1.440e-01  y = (x₅ * (0.035163 - x₅)) - p(x₀ + -0.50753)
12          6.434e+00  3.662e-03  y = (x₅ * (0.030038 - x₅)) + (-0.20386 - p(x₀ + -0.50747))
14          6.095e+00  2.702e-02  y = (sin(p(x₀ - -0.43871)) - p(x₀ + -0.5084)) - (x₅ * x₅)
15          6.087e+00  1.424e-03  y = (sin(p(x₀ - sin(-0.46929))) - p(x₀ + -0.5084)) - (x₅ *...
             

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143254_dPilg4/hall_of_fame.csv


In [56]:
show_results(model2_p_w2)


,complexity,loss,score,equation
3,5,195.229480,3.120413,-20.770073 - (x5 * x5)
4,7,16.436457,1.237337,(x0 * -2.7182167) - (x5 * x5)
5,8,8.643805,0.642659,(p(x0) * -0.9416777) - (x5 * x5)


best: -x5*x5 - sympy_p(x0 - 0.50749236) + sin(x0 + x1 + x1)


### Konfiguracja 3 z operatorem `p` (zakres [-15,15], σ=2)

In [57]:
model3_p_w2 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model3_p_w2.fit(X_p_w2, y_p_w2)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 4.690e+05
Progress: 2706 / 5000 total iterations (54.120%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.636e+03  0.000e+00  y = -95.206
3           4.540e+03  1.017e-02  y = -87.119 - x₀
4           4.423e+03  2.566e-02  y = -73.03 - p(x₀)
5           1.952e+02  3.120e+00  y = -20.77 - (x₅ * x₅)
7           1.644e+01  1.237e+00  y = (x₀ * -2.7181) - (x₅ * x₅)
8           8.646e+00  6.424e-01  y = (p(x₀) * -0.94005) - (x₅ * x₅)
10          6.478e+00  1.443e-01  y = (x₅ * (0.02888 - x₅)) - p(x₀ + -0.50889)
12          6.434e+00  3.409e-03  y = (-0.19421 - p(x₀ - 0.50795)) - (x₅ * (x₅ - 0.030611))
14          6.127e+00  2.443e-02  y = (sin(p(x₀ - -0.57184)) - p(x₀ - 0.50747)) - (x₅ * x₅)
15          6.088e+00  6.429e-03  y = (sin(p(x₀ - sin(-2.6

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143317_ex3135/hall_of_fame.csv


In [58]:
show_results(model3_p_w2)


,complexity,loss,score,equation
3,5,195.229480,3.120413,-20.77007 - (x5 * x5)
4,7,16.436457,1.237337,(x0 * -2.7181113) - (x5 * x5)
5,8,8.643805,0.642659,(p(x0) * -0.9416777) - (x5 * x5)


best: -x5*x5 + sympy_p(x0)*(-0.9416777)


### 5.0 — Zakres [-15, 15], szum $\sigma=5$

In [59]:
X_p_w5, y_p_w5 = make_data_prime(-15, 15, noise_std=5)


### Konfiguracja 1 z operatorem `p` (zakres [-15,15], σ=5)

In [60]:
model1_p_w5 = PySRRegressor(
    binary_operators=["+", "*"],
    unary_operators=["cos", "exp", "sin", "p"],
    maxsize=20,
    extra_sympy_mappings={"p": sympy_p},
    **default_pysr_params,
)
model1_p_w5.fit(X_p_w5, y_p_w5)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 3.530e+05
Progress: 1858 / 5000 total iterations (37.160%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.667e+03  0.000e+00  y = -95.471
3           4.661e+03  4.732e-04  y = x₂ + -95.697
4           4.656e+03  4.891e-04  y = sin(x₂) + -95.469
5           4.034e+02  2.446e+00  y = x₅ * (x₅ * -1.1583)
7           1.733e+02  4.226e-01  y = (x₀ + (x₅ * x₅)) * -1.0994
8           3.152e+01  1.704e+00  y = ((x₅ * x₅) + p(x₀)) * -0.99112
10          2.885e+01  4.432e-02  y = ((x₅ * x₅) + p(x₀ * 0.95491)) * -1.0015
12          2.872e+01  2.212e-03  y = (x₅ * (x₅ * -0.99962)) + (p(x₀ + 0.38213) * -0.91233)
14          2.850e+01  3.842e-03  y = ((x₅ * (x₅ + -0.054485)) * -1.0001) + (p(x₀ + 0.38157)...
                                       * -0.9111)
1

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,20
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20260518_143327_Gm9Ss7/hall_of_fame.csv


In [61]:
show_results(model1_p_w5)


,complexity,loss,score,equation
3,5,403.434700,2.445899,x5 * (x5 * -1.1582632)
5,8,31.519264,1.704235,((x5 * x5) + p(x0)) * -0.99111545
4,7,173.267030,0.422590,(x0 + (x5 * x5)) * -1.0993764


best: (x5*x5 + sympy_p(x0))*(-0.99111545)


### Konfiguracja 2 z operatorem `p` (zakres [-15,15], σ=5)

In [62]:
model2_p_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["cos", "exp", "sin", "log", "p"],
    maxsize=30,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
)
model2_p_w5.fit(X_p_w5, y_p_w5)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.690e+05
Progress: 852 / 5000 total iterations (17.040%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.667e+03  0.000e+00  y = -95.471
3           4.573e+03  9.902e-03  y = -87.375 - x₀
4           4.461e+03  2.440e-02  y = -73.29 - p(x₀)
5           2.109e+02  3.051e+00  y = -21.031 - (x₅ * x₅)
7           3.655e+01  8.764e-01  y = (x₀ * -2.7326) - (x₅ * x₅)
8           3.130e+01  1.551e-01  y = (1.1392 - p(x₀)) - (x₅ * x₅)
9           3.088e+01  1.356e-02  y = ((x₀ * -3.2457) - (x₅ * x₅)) - -5.2358
10          2.901e+01  6.263e-02  y = (-0.12367 - (x₅ * x₅)) - p(x₀ * 0.95615)
12          2.841e+01  1.045e-02  y = (0.569 - (x₅ * x₅)) - p((x₀ + 0.70241) * 0.90922)
14          2.831e+01  1.608e-03  y = (-0.26924 - (x₅ * x₅)) + (1.0661 - p((x₀ 

,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,30
,maxdepth,None
,warmup_maxsize_by,None


Error in callback _flush_stdio (for post_execute), with arguments args (),kwargs {}:


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [63]:
show_results(model2_p_w5)


,complexity,loss,score,equation
3,5,210.936940,3.051560,-21.030697 - (x5 * x5)
4,7,36.553700,0.876388,(x0 * -2.7326171) - (x5 * x5)
5,8,30.918005,0.167444,(p(x0) * -0.9218122) - (x5 * x5)


best: x0*(-2.7326171) - x5*x5
Error in callback _flush_stdio (for post_execute), with arguments args (),kwargs {}:


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

### Konfiguracja 3 z operatorem `p` (zakres [-15,15], σ=5)

In [66]:
model3_p_w5 = PySRRegressor(
    binary_operators=["+", "*", "-", SAFE_POW],
    unary_operators=["exp", "sin", "p"],
    maxsize=15,
    constraints={"safe_pow": (-1, 1)},
    extra_sympy_mappings={"p": sympy_p, **safe_pow_sympy},
    **default_pysr_params,
    verbosity=0,
)
model3_p_w5.fit(X_p_w5, y_p_w5)


/opt/homebrew/Caskroom/miniforge/base/envs/pytorch_env/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


,model_selection,'best'
,binary_operators,"['+', '*', ...]"
,unary_operators,"['exp', 'sin', ...]"
,expression_spec,None
,niterations,100
,populations,50
,population_size,27
,max_evals,None
,maxsize,15
,maxdepth,None
,warmup_maxsize_by,None


Error in callback _flush_stdio (for post_execute), with arguments args (),kwargs {}:


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data

In [67]:
show_results(model3_p_w5)


,complexity,loss,score,equation
3,5,210.93694,3.051560,-21.030697 - (x5 * x5)
4,7,36.55370,0.876388,(x0 * -2.7326171) - (x5 * x5)
5,8,31.30177,0.155108,1.1496086 - (p(x0) + (x5 * x5))


best: x0*(-2.7326171) - x5*x5
Error in callback _flush_stdio (for post_execute), with arguments args (),kwargs {}:


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe2 in position 4095: unexpected end of data